In [1]:
!date

Fri Sep 12 11:15:40 PDT 2025


# 2025-09-12: PBMC - Run starCAT on CD4 T cells - MM Cells
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology

**Main aim**: \
In this notebook, I will use the method described by [Kotliar et al.](https://github.com/immunogenomics/starCAT) to perform NMF decomposition with a pre-computed matrix W called `TCAT.v1`to identify niche T cell states within all CD4 + T cells in the PBMC MM dataset. This notebook uses the kernel defined in the `rna-setup/ac-starcat-envt.yml` file. 
> Tutorial Vigenette: [starCAT_vignette.ipynb](https://github.com/immunogenomics/starCAT/blob/main/Examples/starCAT_vignette.ipynb)

In [2]:
import time
start_time = time.time()

### 1. Imports

In [3]:
import pandas as pd
import numpy as np
import scanpy as sc
import os
import matplotlib.pyplot as plt
import sys
import seaborn as sns
from starcat import starCAT
import sys

sys.path.append("../../00-utilities/functions/python/")
from process_scrna_data import process_adata

In [4]:
import starcat
starcat.available_refs

array(['TCAT.V1', 'MYELOID.GLIOMA.V1', 'BONEMARROW.CD34POS.HSPC.V1'],
      dtype=object)

### 2. Get the TCAT.v1 reference from starCAT

In [5]:
tcat = starCAT(reference='TCAT.V1', cachedir='./cache')

Using reference from starCAT database
Loading reference from existing cache file for reference TCAT.V1


In [6]:
# View first 5 GEPs and 5 genes for default reference spectra (genes x programs)
print(tcat.ref_name)
display(tcat.ref.iloc[:5, :5])

TCAT.V1


,A1BG,AARD,AARSD1,ABCA1,ABCB1
CellCycle-G2M,2.032614,22.965553,17.423538,3.478179,2.297279
Translation,35.445282,0.000000,9.245893,0.477994,0.000000
HLA,18.192997,14.632670,2.686475,3.937182,0.000000
ISG,0.436212,0.000000,18.078197,17.354506,0.000000
Mito,10.293049,0.000000,52.669895,14.615502,3.341488


In [7]:
# This is the information for the scores that will be calculated byt starCAT with the current reference
tcat.score_data

{'scores': {'continuous': [{'name': 'ASA',
    'normalization': 'normalized',
    'columns': ['TIMD4/TIM3', 'ICOS/CD38', 'CTLA4/CD38', 'OX40/EBI3']},
   {'name': 'Proliferation',
    'normalization': 'normalized',
    'columns': ['CellCycle-G2M', 'CellCycle-S', 'CellCycle-Late-S']}],
  'discrete': [{'name': 'ASA_binary',
    'normalization': 'normalized',
    'columns': ['TIMD4/TIM3', 'ICOS/CD38', 'CTLA4/CD38', 'OX40/EBI3'],
    'threshold': 0.0625},
   {'name': 'Proliferation_binary',
    'normalization': 'normalized',
    'columns': ['CellCycle-G2M', 'CellCycle-S', 'CellCycle-Late-S'],
    'threshold': 0.1},
   {'name': 'Multinomial_Label',
    'normalization': 'normalized',
    'file': 'multinomial_lineage_classifier.py',
    'function': 'compute_lineage'}]}}

### 3. Read in the object built for NMFProj

In [8]:
# Load cell x genes counts data
adata = tcat.load_counts('../../../data/rna/pbmc-subsets/pbmc-t-cd4-mm-input-nmf.h5ad')

In [9]:
adata

AnnData object with n_obs × n_vars = 629845 × 31915
    obs: 'batch_id', 'cell_name', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'specimen.specimenGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'tissue', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.batch_id', 'manual.category', 'manual.treatment_dara', 'manual.flu_response', 'aifi_label_l1', 'aifi_celltype_l1', 'aifi_label_l2', 'aifi_celltype_l2', 'aifi_label_l3', 'aifi_celltype_l3', '

### 4. Run starCAT on the raw data & save results

In [10]:
# Run starCAT to compute the usages and scores for the provided data
usage, scores = tcat.fit_transform(adata)

3412 out of 3412 genes in the reference overlap with the query


/home/workspace/environment/starcat/lib/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.0.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [11]:
usage.head()

,CellCycle-G2M,Translation,HLA,ISG,Mito,Doublet-RBC,gdT,CellCycle-S,Cytotoxic,Doublet-Platelet,...,Tfh-2,OX40/EBI3,CD172a/MERTK,IEG3,Doublet-Fibroblast,SOX4/TOX2,CD40LG/TXNIP,Tph,Exhaustion,Tfh-1
barcodes,,,,,,,,,,,,,,,,,,,,,
42c857204e4611ecb4cebaca99a10e91,0.001176,0.045574,0.001497,0.011961,0.023701,0.001137,0.013301,0.001364,0.025394,0.002119,...,0.012894,0.016285,0.051632,0.005851,0.018085,0.007970,0.012845,0.008988,0.009270,0.004667
42c867ec4e4611ecb4cebaca99a10e91,0.000381,0.027772,0.000353,0.029877,0.013666,0.001122,0.014294,0.007270,0.003869,0.001931,...,0.001315,0.004341,0.025522,0.024004,0.010750,0.009615,0.007874,0.000472,0.104929,0.001818
42c8bd004e4611ecb4cebaca99a10e91,0.005237,0.036512,0.005442,0.006428,0.003097,0.000876,0.014945,0.009053,0.006251,0.001816,...,0.002495,0.002160,0.044960,0.017153,0.004573,0.015567,0.018100,0.002788,0.008003,0.003160
42c8d4de4e4611ecb4cebaca99a10e91,0.002391,0.037847,0.002269,0.019440,0.008282,0.001086,0.006754,0.025655,0.015153,0.009690,...,0.005335,0.004205,0.034358,0.015634,0.002810,0.005447,0.017972,0.000785,0.017508,0.005975
42c8e2304e4611ecb4cebaca99a10e91,0.001083,0.036858,0.002758,0.006061,0.001096,0.000941,0.001848,0.013150,0.006279,0.006917,...,0.011686,0.008054,0.020136,0.015827,0.004664,0.004718,0.006190,0.000905,0.015030,0.018509


In [12]:
scores.head()

,ASA,Proliferation,ASA_binary,Proliferation_binary,Multinomial_Label
barcodes,,,,,
42c857204e4611ecb4cebaca99a10e91,0.039174,0.003196,False,False,CD4_EM
42c867ec4e4611ecb4cebaca99a10e91,0.071084,0.008876,True,False,CD4_Naive
42c8bd004e4611ecb4cebaca99a10e91,0.023809,0.031288,False,False,CD4_Naive
42c8d4de4e4611ecb4cebaca99a10e91,0.069268,0.050434,True,False,CD4_CM
42c8e2304e4611ecb4cebaca99a10e91,0.040751,0.015731,False,False,CD4_Naive


In [13]:
usage.to_parquet(
    "../../../data/rna/starcat/pbmc-t-cd4-mm-starcat-usages.parquet",
    engine="fastparquet"
)

scores.to_parquet(
    "../../../data/rna/starcat/pbmc-t-cd4-mm-starcat-scores.parquet",
    engine="fastparquet"
)

## 5. Add in the usages, scores to adata

In [14]:
# Merge usages and scores with cell metadata
adata.obs = pd.merge(
    left=adata.obs, right=usage, how="left", left_index=True, right_index=True
)

# Modify object type to string for plotting in scanpy
scores[scores.columns[scores.columns.str.contains("_binary")]] = scores[
    scores.columns[scores.columns.str.contains("_binary")]
].astype("str")
adata.obs = pd.merge(
    left=adata.obs, right=scores, how="left", left_index=True, right_index=True
)

In [15]:
adata

AnnData object with n_obs × n_vars = 629845 × 31915
    obs: 'batch_id', 'cell_name', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'specimen.specimenGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'tissue', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.batch_id', 'manual.category', 'manual.treatment_dara', 'manual.flu_response', 'aifi_label_l1', 'aifi_celltype_l1', 'aifi_label_l2', 'aifi_celltype_l2', 'aifi_label_l3', 'aifi_celltype_l3', '

In [16]:
starcat_cols = [
    "CellCycle-G2M",
    "Translation",
    "HLA",
    "ISG",
    "Mito",
    "Doublet-RBC",
    "gdT",
    "CellCycle-S",
    "Cytotoxic",
    "Doublet-Platelet",
    "NME1/FABP5",
    "Th22",
    "MAIT",
    "CellCycle-Late-S",
    "Cytoskeleton",
    "Heatshock",
    "Multi-Cytokine",
    "TEMRA",
    "Doublet-Myeloid",
    "Metallothionein",
    "CD4-CM",
    "IEG",
    "CD8-EM",
    "IEG2",
    "CD4-Naive",
    "Treg",
    "Th17-Resting",
    "Poor-Quality",
    "CD8-Naive",
    "RGCC/MYADM",
    "TIMD4/TIM3",
    "Doublet-Plasmablast",
    "BCL2/FAM13A",
    "IL10/IL19",
    "Th2-Activated",
    "Th2-Resting",
    "ICOS/CD38",
    "Doublet-Bcell",
    "Th1-Like",
    "CTLA4/CD38",
    "CD8-Trm",
    "Th17-Activated",
    "Tfh-2",
    "OX40/EBI3",
    "CD172a/MERTK",
    "IEG3",
    "Doublet-Fibroblast",
    "SOX4/TOX2",
    "CD40LG/TXNIP",
    "Tph",
    "Exhaustion",
    "Tfh-1",
    "ASA",
    "Proliferation",
    "ASA_binary",
    "Proliferation_binary",
    "Multinomial_Label",
]
adata.obs[starcat_cols].to_parquet(
    "../../../data/rna/pbmc-subsets/pbmc-t-cd4-mm-starcat-metadata.parquet",  
    engine="fastparquet"
)

In [17]:
end_time = time.time()
print("Total runtime:", end_time - start_time, "seconds")

Total runtime: 89.46194410324097 seconds
